# Workspace stickiness: verbatim-stated trait vs. inferred trait

**Pre-registered, but genuinely exploratory in the sense that the prediction is
two-sided on purpose** — see `prediction.md` (committed before this notebook).
This is a new, separate question from the negation-framing project in
`jacobian-lens-experiment` — it does not attempt to salvage or extend that
project's thesis.

Reuses the already-validated pipeline: same `jlens` library, same
`google/gemma-3-4b-it` checkpoint and lens, same 35-90% depth band. No spider
check or walkthrough reproduction repeated here -- that validation already
happened twice (the negation-framing project, and the Room exploratory
notebook) on this exact model/lens pairing.

**Before running:** `Runtime > Change runtime type > T4 GPU`. Gemma-3-4B-IT is
gated -- accept the license at https://huggingface.co/google/gemma-3-4b-it and
have an HF token stored as a Colab secret named `HF_token` (key icon, left
sidebar; toggle notebook access on).

**What this notebook does:**
1. Loads the model + lens (no re-validation).
2. Tokenizes the target trait word (`generous`) and checks single-token status.
3. Builds two full texts -- direct/verbatim trigger vs. inferred trigger --
   sharing an identical opening, an identical 10-sentence neutral filler
   block, and an identical reintroduction sentence.
4. Locates the checkpoint positions (end of trigger, end of each filler
   sentence, the entity-mention token in the reintroduction, end of the
   reintroduction sentence) programmatically, by counting sentence
   boundaries -- not by guessing token offsets.
5. Reads the target word's rank/logprob at every checkpoint, both conditions,
   within the band -- the pre-registered comparison -- and also dumps the raw
   top-20 J-lens vocabulary at each checkpoint (exploratory, not part of the
   pre-registered comparison, same spirit as the Room notebook).
6. Plots the decay curve for both conditions and saves everything to CSV.

In [ ]:
# Install jlens straight from GitHub -- same as the other two notebooks.
!pip install -q git+https://github.com/anthropics/jacobian-lens.git
!pip install -q huggingface_hub

In [ ]:
# Gemma-3-4B-IT is gated -- reads a Colab secret named 'HF_token' instead of
# the interactive widget (key icon in the left sidebar -> add secret HF_token
# -> toggle notebook access on for this notebook).
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_token"))

## Load the model and lens

Identical to the other two notebooks: same checkpoint, same lens file, no
revision needed.

In [ ]:
import torch
import transformers
import jlens
import pandas as pd

jlens.configure_logging()

GEMMA_MODEL_NAME = "google/gemma-3-4b-it"
GEMMA_LENS_REPO = "neuronpedia/jacobian-lens"
GEMMA_LENS_FILE = "gemma-3-4b-it/jlens/Salesforce-wikitext/gemma-3-4b-it_jacobian_lens.pt"

gemma_hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL_NAME, dtype=torch.bfloat16
).cuda()
gemma_tokenizer = transformers.AutoTokenizer.from_pretrained(GEMMA_MODEL_NAME)
gemma_model = jlens.from_hf(gemma_hf_model, gemma_tokenizer)

gemma_lens = jlens.JacobianLens.from_pretrained(GEMMA_LENS_REPO, filename=GEMMA_LENS_FILE)

n_layers = gemma_model.n_layers
# Same constraint discovered in the negation-framing project: the lens has no
# fitted Jacobian for the final layer.
LAYER_BAND = (0.35, 0.90)
band_layers = [
    l for l in range(n_layers - 1)
    if LAYER_BAND[0] <= l / (n_layers - 1) <= LAYER_BAND[1]
]
print(f"n_layers={n_layers}, band_layers ({len(band_layers)}): {band_layers}")

## Target word: tokenize and check single-token status

Same procedure as the negation-framing project's Step 4: check both bare and
leading-space forms, drop (don't substitute) anything that isn't a single
token.

In [ ]:
TARGET_WORD = "generous"

candidate_forms = [(TARGET_WORD, TARGET_WORD), (TARGET_WORD, " " + TARGET_WORD)]
surviving_targets = []  # (label, token_id)
for word, form in candidate_forms:
    ids = gemma_tokenizer.encode(form, add_special_tokens=False)
    label = f"{word!r} (bare)" if form == word else f"{word!r} (leading-space)"
    if len(ids) == 1:
        surviving_targets.append((label, ids[0]))
        print(f"KEEP  {label:<28} -> 1 token: {ids} ({gemma_tokenizer.decode(ids)!r})")
    else:
        pieces = [gemma_tokenizer.decode([i]) for i in ids]
        print(f"DROP  {label:<28} -> {len(ids)} tokens: {pieces}")

print()
print(f"{len(surviving_targets)} of {len(candidate_forms)} forms survive as single tokens.")

## Build the two conditions

Identical opening and filler/reintroduction; only the trigger sentence
differs. Verbatim from `prediction.md`.

In [ ]:
SHARED_OPENING = "Maria runs the corner bakery in a small town outside Lyon."

DIRECT_TRIGGER = "Maria is generous."

INFERRED_TRIGGER = (
    "Every evening, Maria packs up the bread that didn't sell and drives it "
    "to the shelter across town before deciding what to keep for herself."
)

FILLER_SENTENCES = [
    "The town has one main street lined with plane trees that lose their leaves in early October.",
    "A weekly market sets up in the square every Wednesday morning before the shops open.",
    "The nearest train station is a fifteen-minute walk past the old stone bridge.",
    "Local weather this time of year tends to bring short rain showers in the late afternoon.",
    "The bakery's ovens are relined with fresh brick every few years to keep the heat even.",
    "A small hardware store sits two doors down, run by a family that has owned it for decades.",
    "Traffic on the main road slows considerably during the summer tourist season.",
    "The town council meets on the first Tuesday of each month in the old schoolhouse.",
    "A stray cat has taken to sleeping in the doorway of the shuttered cinema.",
    "The river that runs along the edge of town floods rarely, but the bridge was built high just in case.",
]

REINTRO_SENTENCE = "It is Tuesday. Maria answers the phone."

def build_full_text(trigger):
    parts = [SHARED_OPENING, trigger] + FILLER_SENTENCES + [REINTRO_SENTENCE]
    return " ".join(parts)

CONDITIONS = {
    "direct": build_full_text(DIRECT_TRIGGER),
    "inferred": build_full_text(INFERRED_TRIGGER),
}

for name, text in CONDITIONS.items():
    print(f"=== {name} ===")
    print(text)
    print()

## Locate checkpoint positions

Programmatically, by counting sentence-ending periods -- not by guessing
token offsets. Expected period count: 1 (opening) + 1 (trigger) + 10 (filler)
+ 2 (reintroduction has two sentences) = 14. The entity-mention checkpoint
is the *last* occurrence of "Maria" in the text (the reintroduction's, since
every earlier mention is further back).

In [ ]:
def find_checkpoints(full_text):
    input_ids = gemma_tokenizer(full_text, return_tensors="pt")["input_ids"][0]
    seq_len = input_ids.shape[0]
    position_tokens = gemma_tokenizer.convert_ids_to_tokens(input_ids.tolist())

    period_positions = [i for i, t in enumerate(position_tokens) if t == "."]
    print(f"  found {len(period_positions)} periods (expected 14)")
    assert len(period_positions) == 14, (
        "Period count doesn't match expectation -- tell Claude before trusting "
        "anything downstream. Print position_tokens and inspect by hand."
    )

    checkpoints = {}
    checkpoints["distance_0_trigger_end"] = period_positions[1]
    for i in range(10):
        checkpoints[f"distance_{i+1}_filler_end"] = period_positions[2 + i]
    checkpoints["reintro_sentence_end"] = period_positions[13]

    maria_positions = [
        i for i, t in enumerate(position_tokens)
        if t.strip("▁").lower() == "maria"
    ]
    checkpoints["reintro_entity_mention"] = maria_positions[-1]

    return checkpoints, seq_len, position_tokens


all_checkpoints = {}
for name, text in CONDITIONS.items():
    print(f"=== {name} ===")
    checkpoints, seq_len, position_tokens = find_checkpoints(text)
    all_checkpoints[name] = checkpoints
    for label, pos in checkpoints.items():
        print(f"  {label:28s} -> position {pos:>4d}  token {position_tokens[pos]!r}")
    print()

## Read out the target word at every checkpoint, both conditions

Rank and log-probability of every surviving `generous` form, within the band,
at every checkpoint -- the pre-registered comparison. Also dumps the raw
top-20 J-lens vocabulary at each checkpoint (exploratory only, no target
list, same spirit as the Room notebook) so anything else interesting at the
reintroduction point is visible too.

In [ ]:
rows = []
top20_rows = []

for condition_name, full_text in CONDITIONS.items():
    checkpoints = all_checkpoints[condition_name]
    positions = list(checkpoints.values())
    labels_by_position = {v: k for k, v in checkpoints.items()}

    jlens_logits, _, _ = gemma_lens.apply(
        gemma_model, full_text, layers=band_layers, positions=positions
    )

    for layer in band_layers:
        layer_logits = jlens_logits[layer]  # [len(positions), vocab]
        log_probs = torch.log_softmax(layer_logits.float(), dim=-1)

        for pos_idx, pos in enumerate(positions):
            checkpoint_label = labels_by_position[pos]
            logits_here = layer_logits[pos_idx]
            logprobs_here = log_probs[pos_idx]

            for target_label, token_id in surviving_targets:
                rank = int((logits_here > logits_here[token_id]).sum().item()) + 1
                logprob = float(logprobs_here[token_id].item())
                rows.append({
                    "condition": condition_name,
                    "checkpoint": checkpoint_label,
                    "position": pos,
                    "layer": layer,
                    "depth_frac": layer / (n_layers - 1),
                    "target": target_label,
                    "rank": rank,
                    "logprob": logprob,
                })

            top20 = [gemma_tokenizer.decode([t]) for t in logits_here.topk(20).indices]
            for rank, tok in enumerate(top20, start=1):
                top20_rows.append({
                    "condition": condition_name,
                    "checkpoint": checkpoint_label,
                    "position": pos,
                    "layer": layer,
                    "depth_frac": layer / (n_layers - 1),
                    "rank": rank,
                    "token": tok,
                })

    del jlens_logits, log_probs
    torch.cuda.empty_cache()

results_df = pd.DataFrame(rows)
top20_df = pd.DataFrame(top20_rows)
results_df.to_csv("stickiness_results.csv", index=False)
top20_df.to_csv("stickiness_top20.csv", index=False)
print(f"Wrote {len(results_df)} rows to stickiness_results.csv")
print(f"Wrote {len(top20_df)} rows to stickiness_top20.csv")

## Decay curve: median rank across the band, per checkpoint, per condition

The pre-registered comparison. `distance_0` through `distance_10` trace the
curve; the two `reintro_*` checkpoints are plotted as separate points, not
part of the distance sequence (re-mentioning the entity is a different kind
of event than continued neutral filler).

In [ ]:
summary_rows = []
for (condition, checkpoint, target), group in results_df.groupby(["condition", "checkpoint", "target"]):
    summary_rows.append({
        "condition": condition,
        "checkpoint": checkpoint,
        "target": target,
        "median_rank": group["rank"].median(),
        "best_rank": group["rank"].min(),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("stickiness_summary.csv", index=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)

distance_order = ["distance_0_trigger_end"] + [f"distance_{i}_filler_end" for i in range(1, 11)]
reintro_order = ["reintro_entity_mention", "reintro_sentence_end"]

for target in sorted(summary_df["target"].unique()):
    print(f"=== {target} ===")
    subset = summary_df[summary_df["target"] == target]
    pivoted = subset.pivot(index="checkpoint", columns="condition", values="median_rank")
    pivoted = pivoted.reindex(distance_order + reintro_order)
    print(pivoted.to_string())
    print()

In [ ]:
import matplotlib.pyplot as plt

targets = sorted(summary_df["target"].unique())
fig, axes = plt.subplots(len(targets), 1, figsize=(9, 4 * len(targets)))
if len(targets) == 1:
    axes = [axes]

for ax, target in zip(axes, targets):
    subset = summary_df[summary_df["target"] == target]
    for condition, color in [("direct", "tab:blue"), ("inferred", "tab:orange")]:
        cond_df = subset[subset["condition"] == condition].set_index("checkpoint")
        cond_df = cond_df.reindex(distance_order)
        xs = list(range(11))
        ys = cond_df["median_rank"].tolist()
        ax.plot(xs, ys, marker="o", label=f"{condition} (distance decay)", color=color)

        # reintroduction points plotted just after the decay curve, offset on the x-axis
        reintro_df = subset[subset["condition"] == condition].set_index("checkpoint")
        for i, label in enumerate(reintro_order):
            if label in reintro_df.index:
                ax.plot(11 + i, reintro_df.loc[label, "median_rank"], marker="*", markersize=14,
                        color=color, linestyle="none")

    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_xticks(list(range(11)) + [11, 12])
    ax.set_xticklabels([str(i) for i in range(11)] + ["Maria\n(reintro)", "end\n(reintro)"])
    ax.set_xlabel("distance from trigger (sentences) -- stars are the reintroduction checkpoints")
    ax.set_ylabel("median rank in band\n(lower = stronger)")
    ax.set_title(target)
    ax.legend()

fig.tight_layout()
fig.savefig("stickiness_decay_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Done

Files written:
- `stickiness_results.csv` -- full raw sweep, every (condition, checkpoint, layer, target).
- `stickiness_top20.csv` -- exploratory top-20 vocabulary at every checkpoint, both conditions (not part of the pre-registered comparison).
- `stickiness_summary.csv` -- median/best rank per (condition, checkpoint, target).
- `stickiness_decay_curves.png` -- the plot above.

Download these and send them back the same way as before.